<p> <center> <a href="../start_here_ja.ipynb">Home Page</a> </center> </p>

<div>
    <span style="float: left; width: 33%; text-align: left;"><a href="04_langraph_agent_ja.ipynb">Previous Notebook</a></span>
    <span style="float: left; width: 34%; text-align: center;">
        <a href="01_inference_endpoint_ja.ipynb">1</a>
        <a href="02_introduction_mcp_ja.ipynb">2</a>
        <a href="03_low_level_mcp_ja.ipynb">3</a>
        <a href="04_langraph_agent_ja.ipynb">4</a>
        <a >5</a>
        <a href="06_challenge_ja.ipynb">6</a>
        <a href="bonus_challenge/07_bonus_challenge_ja.ipynb">7</a>
    </span>
    <span style="float: left; width: 33%; text-align: right;"><a href="06_challenge_ja.ipynb">Next Notebook</a></span>
</div>

## 学習目標

このノートブックを終えると、以下ができるようになります：
- low-level SDK と SQLite を使って Movie Database MCP Server を構築する
- `nat mcp client` CLI を使って、クライアントコードを書かずに MCP tool を発見・テストする
- NAT workflow の設定を YAML で定義し、MCP tool を NVIDIA NIM endpoint に接続する
- LangGraph の intent classifier を NAT workflow 内にラップして、クエリが agent に到達する前にルーティングする
- Phoenix を使って NAT の telemetry を設定し、エンドツーエンドの tracing を行う

## はじめに

前回のノートブックでは、LLM・tool・制御フローを Python で配線して LangGraph agent を手作業で構築しました。これは動作しますが、スケールしません：新しい agent を作るたびにグルーコードが増え、agent 間で tool を共有したり LLM バックエンドを切り替えたりする標準的な方法もありません。

**NVIDIA NeMo Agent Toolkit (NAT)** はまさにこれを解決するために設計されています。フレームワーク非依存のツールキットで、agent・LLM・tool を YAML で宣言し、Model Context Protocol (MCP) 経由で接続し、単一の CLI (`nat run`, `nat serve`, `nat eval`) で実行します。同じ YAML config で ReAct agent も LangGraph workflow も multi-agent システムも実行でき、agent コードに触れずに Nemotron を GPT や Llama に差し替えることもできます。

このノートブックでは、NAT 駆動の小さくも完全なシステムをエンドツーエンドで構築します：

1. **SQLite ベースの Movie Database**(3 テーブルに 117 本の映画、IMDB レーティング・興行収入・ジャンルを格納)
2. それを **low-level MCP Server** として公開し、`search_movies` tool を提供する
3. それを完全に YAML で設定された **NAT ReAct agent** で利用する
4. **LangGraph intent classifier** でゲートを設け、ドメイン内のクエリのみを agent にルーティングし、それ以外は拒否する
5. **Phoenix telemetry** でエンドツーエンドにトレースする

最終的に、NVIDIA の本番 agentic システムを支えているのと同じアーキテクチャパターン(ノートブック規模ですが)が手元にできあがります。

## 環境のセットアップ

何かを構築する前に、LLM バックエンドが必要です。NAT は `build.nvidia.com` でホストされたものでも、自分の GPU でローカルに動作するものでも、任意の NVIDIA NIM endpoint と通信できます。速度のためデフォルトではクラウド endpoint を使用しますが、完全オフラインで動かしたい場合はローカル経路も以下に記載しています。

### Cloud Endpoint

以下のセルは `NVIDIA_API_KEY` が未設定の場合にキー入力を求めます。まだキーを生成していない場合は [Notebook 1](01_inference_endpoint_ja.ipynb) を参照してください。

In [ ]:
import os
import getpass
import warnings
warnings.filterwarnings("ignore")
if not os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"):
    nvapi_key = getpass.getpass("Enter your NVIDIA API key: ")
    assert nvapi_key.startswith("nvapi-"), f"{nvapi_key[:5]}... is not a valid key"
    os.environ["NVIDIA_API_KEY"] = nvapi_key
    os.environ["NGC_API_KEY"] = nvapi_key

### Local Endpoint

クラウド endpoint ではなくローカルで LLM を実行したい場合は、自分のマシン上で動作している任意の NIM コンテナに NAT を向けることができます。NIM コンテナの起動方法は [Notebook 1](01_inference_endpoint_ja.ipynb) を参照してください — 後で必要な変更は workflow YAML の `base_url` を入れ替えることだけです。

## データベースを確認する

データを MCP tool として公開する前に、`movie.sqlite` の中身を確認しておきましょう。データベースには 3 つのテーブルがあります：

## 行のプレビュー

IMDB テーブルの一部の行をプレビューして、データ構造を把握します。各映画には ID、タイトル、レーティング、投票数、予算、上映時間が含まれています。

In [ ]:
import sqlite3
import json

DB_PATH = "data/movie.sqlite"

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cursor = conn.cursor()
cursor.execute("SELECT Movie_id, Title, Rating, TotalVotes, Budget, Runtime FROM IMDB LIMIT 5")
rows = [dict(row) for row in cursor.fetchall()]
conn.close()

print(json.dumps(rows, indent=2))

### データベースを Helper クラスでラップする

`MovieDB` という helper クラスを作成し、SQLite クエリをラップして MCP 型の `TextContent` オブジェクトとして結果を返します。このパターンによりデータベースロジックを MCP server から分離し、独立してテストしやすくなります。

`%%writefile` magic はセル内容を Python ファイルに書き出します。これによりノートブックからも MCP server からも import できるようになります。

In [ ]:
%%writefile movie_db.py
import sqlite3
import json                                                                                                                                                                             
from typing import List
from pathlib import Path                                                                                                                                                                
import mcp.types as types

DB_PATH = "data/movie.sqlite"

class MovieDB:
  def __init__(self, db_path):
      self.db_path = str(Path().resolve().joinpath(db_path))

  def _search_movies(
      self,
      title: str | None,
      min_rating: float | None,
      max_rating: float | None,
      limit: int = 20,
  ) -> List[types.TextContent]:
      """Search movies in the IMDB table by title and/or rating range.

      Args:
          title: Partial or full movie title to search for.
          min_rating: Minimum IMDB rating.
          max_rating: Maximum IMDB rating.
          limit: Max number of results to return.
      """

      conn = sqlite3.connect(self.db_path)
      cursor = conn.cursor()

      query = """
          SELECT Movie_id, Title, Rating, TotalVotes, Budget, Runtime
          FROM IMDB
      """
      params = []
      conditions = []

      if title is not None:
          conditions.append("Title LIKE ?")
          params.append(f"%{title}%")

      if min_rating is not None:
          conditions.append("Rating >= ?")
          params.append(min_rating)

      if max_rating is not None:
          conditions.append("Rating <= ?")
          params.append(max_rating)

      if conditions:
          query += " WHERE " + " AND ".join(conditions)

      query += " ORDER BY Rating DESC LIMIT ?"
      params.append(limit)

      try:
          cursor.execute(query, params)
          results = cursor.fetchall()

          output = []
          for row in results:
              output.append(
                  {
                      "movie_id": row[0],
                      "title": row[1],
                      "rating": row[2],
                      "total_votes": row[3],
                      "budget": row[4],
                      "runtime": row[5],
                  }
              )

      except sqlite3.Error as e:
          conn.close()
          raise e

      finally:
          conn.close()

      return [types.TextContent(
          type="text",
          text=json.dumps(output)
      )]

In [ ]:
from movie_db import MovieDB

In [ ]:
movie_db = MovieDB(DB_PATH)

### Helper クラスのテスト

MCP server を構築する前に、`MovieDB` クラスがいくつかの異なるクエリパターンで正しく動作することを確認します。各呼び出しは MCP の `TextContent(type="text", text=…)` を 1 要素のリストで返します。`text` は JSON エンコードされた結果リストで、これが反対側の MCP client が期待する形状です。

**タイトルによる検索** — タイトルに "Dark" が含まれる映画を検索：

In [ ]:
movie_db._search_movies(title="Dark", min_rating=None, max_rating=None)

**最低レーティングによる検索** — 8.5 以上のレーティングの映画：

In [ ]:
movie_db._search_movies(title=None, min_rating=8.5, max_rating=None, limit=5)

**複合フィルター** — タイトルに "The" を含み、レーティングが 8.0 以上の映画：

In [ ]:
movie_db._search_movies(title="The", min_rating=8.0, max_rating=None, limit=3)

**フィルターなし** — レーティングの高い上位 5 本の映画を返す：

In [ ]:
movie_db._search_movies(title=None, min_rating=None, max_rating=None, limit=5)

## MCP サーバーの構築

`MovieDB` クラスをlow level MCP サーバー（HTTP transport）でラップします（[Notebook 3](03_low_level_mcp_ja.ipynb) で構築したものと同様）。サーバーは `search_movies` という 1 つの tool を公開し、クライアントが発見・呼び出しできるようにします。Starlette と Uvicorn を使って MCP endpoint を `http://127.0.0.1:8000/mcp` でサービングします。

### Server ファイルの中身

次のセルは、movie database を HTTP で公開する MCP server を書き出します。ファイルの最後のセクションは、low-level MCP server を Starlette web アプリケーションに接続しています：

1. **`handle_streamable_http`** は MCP endpoint の ASGI リクエストハンドラーです。Starlette が各リクエストを `scope`, `receive`, `send` を通じて渡し、ハンドラーはそれらのオブジェクトを `session_manager.handle_request(...)` に委譲します。これは Streamable HTTP MCP プロトコルを実装しています。

2. **`lifespan`** は MCP セッションマネージャーの起動と停止を制御します。`async with session_manager.run()` により、Starlette アプリが動作している間 MCP transport を有効に保ち、server が停止する際にクリーンに終了させます。

3. **`starlette_app`** は MCP ハンドラーを `/mcp` にマウントします。これが NAT や他の MCP client が接続する HTTP endpoint で、デフォルトポートは `8000`、URL は `http://127.0.0.1:<MCP_PORT>/mcp` です。

In [ ]:
%%writefile movie_server.py                                                                                                                                                             
import sys
import os
from typing import Any                                                                                                                                                                  
from mcp.server.lowlevel import Server                                                                                                                                                  
from mcp.server.streamable_http_manager import StreamableHTTPSessionManager
from starlette.applications import Starlette                                                                                                                                            
from starlette.routing import Mount
from starlette.types import Receive, Scope, Send
import mcp.types as types
import json
import contextlib
import logging
from collections.abc import AsyncIterator
import uvicorn
from movie_db import MovieDB

logger = logging.getLogger(__name__)

DB_PATH = "data/movie.sqlite"


def main(db_path: str = DB_PATH):
  movie_db = MovieDB(db_path)
  mcp = Server("movie-db")

  @mcp.list_tools()
  async def handle_list_tools() -> list[types.Tool]:
      return [
          types.Tool(
      name="search_movies",                                                                                                                                                               
      description="Search movies by title and/or rating range. Results are sorted by rating descending. To get the top N highest rated movies, just set limit=N without any rating filters. Rating scale is 1.0 to 10.0. In this dataset, ratings range from 7.5 to 8.8.",
      inputSchema={
          "type": "object",
          "properties": {
              "title": {"type": "string", "description": "Partial or full movie title to search for"},
              "min_rating": {"type": "number", "description": "Minimum IMDB rating (1.0 to 10.0)"},
              "max_rating": {"type": "number", "description": "Maximum IMDB rating (1.0 to 10.0)"},
              "limit": {"type": "integer", "description": "Max results to return (default 20)"},
          },
      },
      ),
      ]

  @mcp.call_tool()
  async def handle_call_tool(name: str, args: dict[str, Any] | None):
      args = args or {}

      if name == "search_movies":
          return movie_db._search_movies(
              title=args.get("title"),
              min_rating=args.get("min_rating"),
              max_rating=args.get("max_rating"),
              limit=args.get("limit", 20),
          )
      else:
          return [types.TextContent(
              type="text",
              text=json.dumps({"error": f"Unknown tool: {name}"})
          )]

  session_manager = StreamableHTTPSessionManager(
      app=mcp,
      event_store=None,
      json_response=True,
      stateless=True,
  )

  async def handle_streamable_http(
      scope: Scope, receive: Receive, send: Send
  ) -> None:
      await session_manager.handle_request(scope, receive, send)

  @contextlib.asynccontextmanager
  async def lifespan(app: Starlette) -> AsyncIterator[None]:
      async with session_manager.run():
          logger.info("Movie DB MCP server started!")
          try:
              yield
          finally:
              logger.info("Movie DB MCP server shutting down...")

  starlette_app = Starlette(
      debug=True,
      routes=[
          Mount("/mcp", app=handle_streamable_http),
      ],
      lifespan=lifespan,
  )
  port = int(os.environ.get("MCP_PORT", "8000"))
  uvicorn.run(starlette_app, host="127.0.0.1", port=port)


if __name__ == "__main__":
  db_path = sys.argv[1] if len(sys.argv) > 1 else DB_PATH
  main(db_path)

MCP サーバーをバックグラウンドで起動し、続くセルからテストできるようにします。

In [ ]:
import os, subprocess, time

MCP_PORT = int(os.environ["MCP_PORT"])

log = open("mcp_server.log", "w")
mcp_server_process = subprocess.Popen(
    ["python", "movie_server.py", "data/movie.sqlite"],
    stdout=log,
    stderr=subprocess.STDOUT,
)

time.sleep(2)  # give the server a moment to bind the port
print(f"Server started on http://127.0.0.1:{MCP_PORT}/mcp (PID: {mcp_server_process.pid})")
print("Logs: mcp_server.log")

> **Tip:** server が起動に失敗した場合は、作業ディレクトリの `mcp_server.log` でエラーを確認してください。最も多い原因は、前回のセッションでポートがまだ使用中のままになっていることです。

## NAT CLI でテストする

`nat` は NeMo Agent Toolkit に同梱されている単一のエントリポイントです。Python モジュールを import してランタイムを手で配線する代わりに、YAML config を `nat` に指定すると、そのファイルで宣言された LLM、tool server、agent、telemetry を読み込んでくれます。同じバイナリで agentic システムのライフサイクル全体をカバーできます：

| コマンド | 動作 |
|---------|-------------|
| `nat run --config_file workflow.yml --input "..."` | workflow を 1 つの入力に対して一度実行する(以下で使用します)。 |
| `nat serve --config_file workflow.yml` | workflow を HTTP/SSE API として公開し、他のサービスから呼び出せるようにする。 |
| `nat eval --config_file workflow.yml` | 組み込みの評価ハーネスで workflow をデータセットに対して実行する。 |
| `nat mcp client tool list --url ...` | 任意の MCP server に接続し、`tools/list` RPC を実行して、すべての tool を JSON schema 付きで表示する。 |
| `nat mcp client tool call <name> --url ... --json-args '{...}'` | 名前付き MCP tool を呼び出し、生のレスポンスを表示する。 |
| `nat info components` | YAML config で使用可能なすべての plugin (`_type`) を一覧表示する — agent、LLM、retriever、exporter など。 |

今必要なのは `nat mcp client` サブコマンドです：[Notebook 3](03_low_level_mcp_ja.ipynb) で手書きした MCP client を 2 つのワンライナーで置き換えられます。これらを使って、(1) server が `search_movies` を定義した schema 通りに公開していること、(2) 上記の直接呼び出し `movie_db._search_movies(...)` と同じ行を返していること、を確認します。どちらかが失敗すると、次のセクションの agent も同じ理由で失敗するので、ここで捕まえておく方が良いです。

In [ ]:
!nat mcp client tool list --url http://127.0.0.1:{MCP_PORT}/mcp/

CLI から `search_movies` tool を直接呼び出して、期待通りの結果が返されることを確認します。

In [ ]:
!nat mcp client tool call search_movies \
      --url http://127.0.0.1:${MCP_PORT}/mcp/ \
      --json-args '{"min_rating": 8.5, "limit": 5}'

## NAT Workflow を定義する

NAT workflow は YAML 設定ファイルで定義します。Workflow config では主に 3 つのセクションを指定します：
- **`function_groups`**：外部 tool 接続(ここでは MCP server)
- **`llms`**：使用する LLM バックエンド(ここでは Nemotron を持つ NVIDIA NIM)
- **`workflow`**：agent の種別と tool が LLM にどう接続されるか

まずは movie search tool を呼び出せるシンプルな ReAct agent から始めます。以下の `parse_agent_response_max_retries: 3` の設定は、agent が自分の出力を有効な ReAct の `Action:` / `Thought:` ブロックとしてパースできなかった場合に最大 3 回まで retry するように指示するものです — LLM がフォーマットから逸脱するときによく発生する failure mode です。

In [ ]:
%%writefile movie_workflow.yml                                                                                                                                                          
function_groups:                                                                                                                                                                        
    mcp_movies:                                                                                                                                                                           
      _type: mcp_client                                                                                                                                                                   
      server:                                                                                                                                                                             
        transport: streamable-http
        url: "http://127.0.0.1:${MCP_PORT}/mcp/"

llms:
    nim_llm:
      _type: nim
      model_name: nvidia/nemotron-3-nano-30b-a3b
      base_url: https://integrate.api.nvidia.com/v1
      api_key: ${NVIDIA_API_KEY}
      temperature: 0.0
      max_tokens: 1024

workflow:
    _type: react_agent
    tool_names:
      - mcp_movies
    llm_name: nim_llm
    verbose: true
    parse_agent_response_max_retries: 3

シンプルな映画クエリでワークフローを実行します。

In [ ]:
!nat run --config_file movie_workflow.yml --input "movies rated over 8.5"

### Agent の境界を探る

ドメイン内のクエリは機能しました。しかし、この素の ReAct agent には「ドメイン内」と判定する**ゲートがありません** — movie database と何の関係もない質問でも `search_movies` を呼び出そうとするのを止めるものは何もありません。明らかにドメイン外の 2 つのプロンプトと、ドメイン内の事実質問 1 つで動作を確認してみましょう。

In [ ]:
!nat run --config_file movie_workflow.yml --input "How do I cook pasta?"

In [ ]:
!nat run --config_file movie_workflow.yml --input "What is the capital of France?"

In [ ]:
!nat run --config_file movie_workflow.yml --input "what is the rating of the movie The Dark Knight Rises?"

実行内容次第で、3 つの入力に対して以下のような複数の振る舞いが見られます：

- **拒否** — agent が正しく断る(良いことですが、保証はない)
- **不要な tool 呼び出し** — `title="pasta"` や `title="France"` で `search_movies` を呼び出してしまい、token と tool latency を無駄にする
- **ハルシネーション** — tool 呼び出しを一切せずに勝手な答えをでっち上げる
- **retry の churn** — ReAct ループが `parse_agent_response_max_retries` の上限を使い切ってから諦める

これは NAT や agent のバグではありません — 任意のユーザー入力が tool 呼び出しループに直接流れ込むことを許せば、こうなるのが当然です。最後のドメイン内クエリ(*Dark Knight Rises*)はおそらくうまくいっているはずですが、3 つすべてを実行し終えた時点で、各クエリが*なぜ*成功または失敗したのかを冗長なログを読み返すしか確認方法がありません。

本番システムには 2 つのものが必要です：

1. **決定論的なゲート** — すべてのクエリをまず分類し、ドメイン内のものだけが agent に到達する。
2. **何が起きたかを見る手段** — フィルタリングと比較ができる構造化された trace、ログの壁ではなく。

次の 2 つのセクションでその両方を加えます：LangGraph intent classifier(ゲート)と Phoenix telemetry(ゲートと agent が実際に何をしたかを見るビュー)です。

## LangGraph で Intent Classifier を構築する

先ほど観察した動作はよくあるものです：tool を使う agent は、明らかにそのドメインに属さないクエリに対しても自分の tool を使おうとします。対策は、agent の前に**安価で決定論的な classifier** を置き、関連する質問だけが完全な ReAct ループを経るようにすることです。

ここではルーティングが小さなステートマシンなので、LangGraph がぴったり当てはまります：

これを単なるプロンプトエンジニアリング以上のものにしているのは 2 点です：

1. **関心の分離** — classifier LLM は質問だけを見ており、tool の説明を一切見ないので、「すべての入力に tool が必要」と誤認させるのがはるかに困難になります。
2. **コスト制御** — 拒否されたクエリは agent ループに到達せず、MCP にも触れません。`reject` は LLM コストがゼロのハードコードされた文字列です。

### ファイルの中身

次のセルは `movie_intent_classifier.py` を書き出します。これは movie ReAct agent の前段に位置する小さな LangGraph workflow です。役割は、各ユーザーの質問を 2 つの経路のいずれかにルーティングすることです：

- 映画関連の質問は `movie_agent` に送り、MCP movie database tool を使用させる。
- 映画と関係ない質問は `reject` に送り、movie agent や MCP server を呼び出さずに固定のレスポンスを返す。

このファイルは 6 つの部分に分けて読んでください：

1. **State 型: `ConversationState`**
    ```python
    class ConversationState(TypedDict):
      messages: Annotated[list, add_messages]
    ```
    これは LangGraph node 間で共有される state を定義しています。フィールドは `messages` だけで、会話を格納します。`add_messages` は LangGraph の reducer で、node が `{"messages": [...]}` を返したとき、リストを置き換えるのではなく、既存のリストに新しいメッセージを追加します。

2. **NAT からのランタイムハンドル**
    ```python
    chat_llm = SyncBuilder.current().get_llm("nim_llm", wrapper_type=LLMFrameworkEnum.LANGCHAIN)
    movie_agent = SyncBuilder.current().get_function("movie_agent")
    ```
    これらの行は、NAT の YAML ファイル内ですでに設定済みのオブジェクトを取得しています。`nim_llm` は分類に使う NIM ベースの chat モデルで、`movie_agent` は MCP movie tool を知っている ReAct agent です。Python ファイル側で client を作成したり API キーを直接読んだりはしません。NAT がそれをまとめてセットアップします。

3. **`classify_node`：質問がここに属するかを判定する**
    この node はユーザーのメッセージを classifier LLM に送ります。厳格な system prompt で、LLM は `in_domain` または `out_of_domain` のいずれかでのみ答えなければなりません。判定結果は assistant メッセージとして `messages` に書き戻されるので、ルーティングの判断が LangGraph の state にも、後で Phoenix の trace にも見える形で残ります。

4. **`route_intent`：次の node を選ぶ**
    この関数は最後のメッセージから classifier の判定を読み取ります。判定が厳密に `in_domain` であれば、LangGraph は `movie_agent` にルーティングします。それ以外の値であれば `reject` にルーティングします — 想定外の応答が来た場合の安全なフォールバックです。

5. **Action node：`movie_agent_node` と `reject_node`**
    `movie_agent_node` は NAT ReAct agent を元のユーザークエリで呼び出すので、`async def` で `await movie_agent.ainvoke(...)` を使います。`reject_node` は LLM も tool 呼び出しも不要で、即座にハードコードされたレスポンスを返します。

6. **Graph の配線と compile ステップ**
    ファイルの最後では LangGraph を作成しています：`START -> classify`、そこから条件付き edge で `movie_agent` か `reject` のどちらかに、最後に両方の経路が `END` で終了します。`intent_graph = graph.compile()` で実行可能な workflow が作られ、これを NAT が YAML ファイルから読み込みます。

In [ ]:
%%writefile movie_intent_classifier.py
from typing import Annotated
from typing_extensions import TypedDict                                                                                                                                                 
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

from nat.builder.framework_enum import LLMFrameworkEnum
from nat.builder.sync_builder import SyncBuilder

chat_llm = SyncBuilder.current().get_llm("nim_llm", wrapper_type=LLMFrameworkEnum.LANGCHAIN)
movie_agent = SyncBuilder.current().get_function("movie_agent")


class ConversationState(TypedDict):
  messages: Annotated[list, add_messages]


def classify_node(state: ConversationState):
  response = chat_llm.invoke([
      {"role": "system", "content": (
          "You are an intent classifier for a movie database assistant. "
          "The database contains IMDB movie data: titles, ratings, votes, budgets, and runtimes. "
          "Classify the user's question as either 'in_domain' or 'out_of_domain'. "
          "Reply with ONLY one word.\n\n"
          "in_domain: questions about movies, ratings, top movies, searching movies by title, "
          "comparing movie ratings, budgets, runtimes.\n\n"
          "out_of_domain: anything unrelated to movies."
      )},
      *state["messages"]
  ])
  raw = response.content.strip().lower()
  return {"messages": [{"role": "assistant", "content": raw}]}


def route_intent(state: ConversationState):
  last = state["messages"][-1].content.strip().lower()
  if last == "in_domain":
      return "movie_agent"
  return "reject"


async def movie_agent_node(state: ConversationState):
  user_query = state["messages"][0].content
  result = await movie_agent.ainvoke(user_query)
  return {"messages": [{"role": "assistant", "content": str(result)}]}


def reject_node(state: ConversationState):
  return {"messages": [{"role": "assistant", "content": "Sorry, I can only help with movie-related questions."}]}


graph = StateGraph(ConversationState)
graph.add_node("classify", classify_node)
graph.add_node("movie_agent", movie_agent_node)
graph.add_node("reject", reject_node)

graph.add_edge(START, "classify")
graph.add_conditional_edges("classify", route_intent, {"movie_agent": "movie_agent", "reject": "reject"})
graph.add_edge("movie_agent", END)
graph.add_edge("reject", END)

intent_graph = graph.compile()

classifier はまた、後で telemetry を仕掛ける際の自然な場所も提供します：各 node (`classify`, `movie_agent`, `reject`) はそれぞれ自分の span を発します。なので、次のセクションでは、各クエリがどの経路を通ったかを推測ではなく*目で*見ることができます。

In [ ]:
%%writefile movie_intent_workflow.yml                                                                                                                                                   
function_groups:                                                                                                                                                                        
    mcp_movies:                                                                                                                                                                           
      _type: mcp_client
      server:                                                                                                                                                                             
        transport: streamable-http
        url: "http://127.0.0.1:${MCP_PORT}/mcp/"

llms:
    nim_llm:
      _type: nim
      model_name: nvidia/nemotron-3-nano-30b-a3b
      base_url: https://integrate.api.nvidia.com/v1
      api_key: ${NVIDIA_API_KEY}
      temperature: 0.0
      max_tokens: 1024

functions:
    movie_agent:
      _type: react_agent
      tool_names:
        - mcp_movies
      llm_name: nim_llm
      verbose: true
      parse_agent_response_max_retries: 3

workflow:
    _type: langgraph_wrapper
    graph: movie_intent_classifier.py:intent_graph

In [ ]:
!nat run --config_file movie_intent_workflow.yml --input "How do I cook pasta?"

In [ ]:
!nat run --config_file movie_intent_workflow.yml --input "What is the capital of France?"

In [ ]:
!nat run --config_file movie_intent_workflow.yml --input "what is the rating of the movie The Dark Knight Rises?"

## Phoenix を使った Telemetry

Agent は従来のソフトウェアより debug が困難です。1 つのユーザークエリが複数の LLM 呼び出し、tool 呼び出し、推論ステップに分岐し、それぞれに latency、コスト、failure mode があります。Agent が間違った答えを返したとき、`print()` は*何が*出てきたかは教えてくれますが、*なぜ*そうなったかは教えてくれません：どの tool がおかしな結果を返したのか？どの LLM ステップがハルシネーションしたのか？ReAct ループが coherent なプランに到達する前に retry を使い切ったのか？

これを解決するのが **observability** です。散らばった `print` 文の代わりに、各実行ステップを構造化された **span** として発信します — 入力、出力、タイミング、トークン数、エラーをすべて捕捉し、trace ツリーとして可視化します。その trace が debugger、profiler、評価データセットを兼ねるものになります。

### なぜ Phoenix なのか？

[Arize Phoenix](https://phoenix.arize.com/) は 2 つの open standard を基盤としたオープンソースの LLM observability プラットフォームです：

- **OpenTelemetry (OTel)** — 分散トレーシングの業界標準ワイヤープロトコル。本番のマイクロサービスインフラに広くデプロイ済みで、データモデルがエンジニアリングチームによく理解されている。
- **OpenInference** — OTel の上に重ねられたセマンティック規約。LLM/agent の概念(chat メッセージ、tool 呼び出し、retrieval、embedding、reranker スコア)を span 属性として*どう*表現するかを標準化する。

NAT と Phoenix の両方が OpenInference に対応しているので、telemetry はポータブルです：NAT ReAct agent も LangGraph workflow も CrewAI crew も、同じ trace viewer が独自のアダプタなしで描画します。さらにワイヤーフォーマットが OTel なので、exporter の設定を入れ替えるだけで、同じ span を任意の OTel 互換バックエンドに送れます — agent コードを変更する必要はありません。

### NAT が自動で計装するもの

workflow YAML で Phoenix を有効にすると、スタックの各層が span を発します — デコレーターも SDK 呼び出しも、`start_span` / `end_span` の管理コードも不要です：

| 層 | span 属性として捕捉される内容 |
|-------|-----------------------------|
| **Workflow エントリ** | ユーザー入力、最終出力、実時間レイテンシ |
| **LangGraph node** | node 名、state の入出力、ルーティング判定 |
| **ReAct イテレーション** | Thought、選択した action、observation、イテレーションインデックス |
| **LLM 呼び出し** | モデル名、完全なメッセージ履歴、completion、prompt/completion のトークン数、finish reason |
| **Tool 呼び出し(MCP 含む)** | tool 名、JSON 引数、戻り値、duration、例外 |

結果は入れ子のウォーターフォールになります：最外側の span がユーザークエリ、その子がグラフ node、その子が LLM や tool 呼び出し、最下層がワイヤー上の生の MCP リクエストです。

### Phoenix Observability Server を起動する

agent を構築・実行する前に、[Arize Phoenix](https://docs.arize.com/phoenix) を起動します — LLM アプリケーション向けのオープンソース observability プラットフォームです。Phoenix は NeMo Agent Toolkit workflow からの **trace** を取り込み、すべての LLM 呼び出し、tool 呼び出し、agent 判断ステップを視覚的な UI で確認できるようにします。

In [ ]:
import os, subprocess

PHOENIX_PORT = int(os.environ["PHOENIX_PORT"])

log = open("phoenix.log", "w")
phoenix_process = subprocess.Popen(
    ["phoenix", "serve"],
    stdout=log,
    stderr=subprocess.STDOUT,
)

print(f"Server started on http://127.0.0.1:{PHOENIX_PORT} (PID: {phoenix_process.pid})")
print(f"Access via port-forwarding at http://localhost:6006")
print("Logs: phoenix.log")


### このセルがやっていること

- `phoenix serve` をバックグラウンドの subprocess として起動し、stdout/stderr を `phoenix.log` にリダイレクトする。
- notebook の kernel を解放しておくので、Phoenix がバックグラウンドで trace を集めている間も他のセルを実行し続けられる。

### Phoenix UI を開く

server が起動したら、ブラウザで **[http://localhost:6006](http://localhost:6006)** にアクセスして Phoenix ダッシュボードに入ります。コンテナからローカルマシンへの port-forwarding は bootcamp 環境が自動で処理してくれます — 追加のセットアップは不要です。

このノートブックの後半で、agent 実行が生成した trace を確認するためにこの UI に戻ってきます。

> **Tip:** server が起動に失敗した場合は、作業ディレクトリの `phoenix.log` でエラーを確認してください。最も多い原因は、前回のセッションでポートがまだ使用中のままになっていることです。

### Trace を読む

Phoenix は port-forwarding の設定済みで、ポート **6006** で起動しています。以下で workflow を実行すると、新しい trace が [http://localhost:6006/](http://localhost:6006/) の `movie-mcp` プロジェクトの下に表示されます。任意の trace をクリックすると以下が見えます：

- **Span ツリー(左)** — 入れ子の実行：workflow → classify → movie_agent → LLM → MCP tool
- **Span 詳細(右)** — chat メッセージのインライン描画、tool 引数、JSON の戻り値、トークン数、レイテンシ、エラー時のスタックトレース
- **プロジェクトビュー** — 集約された latency / token / cost のヒストグラムと、全実行を横断できるフィルタ可能な検索

これにより可能になる典型的なデバッグパターン：

- **答えが間違っている?** span ツリーを下に辿り、出力にすでに誤りが含まれている最初のステップを探します — その span がバグの場所です。
- **応答が遅い?** span を duration でソートします。プロンプトが大きすぎる LLM 呼び出しや、サイレントな retry ループはすぐに浮かび上がります。
- **Tool エラー?** span を `status = error` でフィルタすれば、セッション内のすべての失敗 MCP 呼び出しが見られます。
- **トークン爆発?** プロジェクトレベルのトークン集計により、自前で計装しなくても実行間でプロンプトの regression が可視化されます。

これまでに構築したものすべて — MCP tool server、NIM ベースの ReAct agent、LangGraph intent classifier、Phoenix exporter — が、1 つの workflow YAML で配線されます：

- **`function_groups.mcp_movies`** — MCP server (tools)
- **`general.telemetry`** — 上で設定した通りの Phoenix tracing
- **`llms.nim_llm`** — NVIDIA NIM バックエンド
- **`functions.movie_agent`** — ReAct agent。今度は**名前付き関数**として宣言されているので、LangGraph node が `SyncBuilder.current().get_function("movie_agent")` で名前から取得できる
- **`workflow`** — トップレベルの `langgraph_wrapper` が `movie_intent_classifier.py` の `intent_graph` を指している

注目すべきは YAML の中に**ない**もの：明示的な計装コードがありません。先ほどの workflow ではトレース無しで動いていた同じ `movie_agent` が、ファイル上部で `general.telemetry` が設定されているというだけで、すべての LLM 呼び出しと tool 呼び出しに対して完全な Phoenix span を発するようになっています。


### Workflow 内で Phoenix を設定する

Telemetry は workflow YAML の中のもう 1 つのセクションに過ぎません。`general.telemetry` ブロックが実行全体で exporter を有効にし、すべての関数、LLM、tool が自動的にそれを継承します：

```yaml
general:
  telemetry:
    tracing:
      phoenix:
        _type: phoenix
        endpoint: http://localhost:${PHOENIX_PORT}/v1/traces
        project: movie-mcp
```

注目に値する点が 3 つあります：

1. **`_type: phoenix`** — tracing provider plugin。NAT の telemetry レイヤーは pluggable なので、同じ span を別の OTel 互換バックエンドにルーティングするのは YAML のみの変更で済みます。
2. **`endpoint`** — Phoenix の OTLP-HTTP trace 取り込み endpoint。port は `$PHOENIX_PORT` から注入されるので、同じ YAML がこのコンテナでも、ラップトップでも、CI でも編集なしで動作します。
3. **`project`** — 論理的な名前空間。`movie-mcp` でタグ付けされたすべての trace が同じ Phoenix プロジェクトビューに入るので、実行間の比較、モデル間の比較、project でフィルタすることでの regression の特定が可能になります。

In [ ]:
%%writefile movie_intent_workflow_tracing.yml                                                                                                                                                   
function_groups:                                                                                                                                                                        
    mcp_movies:                                                                                                                                                                           
      _type: mcp_client
      server:                                                                                                                                                                             
        transport: streamable-http
        url: "http://127.0.0.1:${MCP_PORT}/mcp/"

general:
  telemetry:
    tracing:
      phoenix:
        _type: phoenix
        endpoint: http://localhost:${PHOENIX_PORT}/v1/traces
        project: movie-mcp

llms:
    nim_llm:
      _type: nim
      model_name: nvidia/nemotron-3-nano-30b-a3b
      base_url: https://integrate.api.nvidia.com/v1
      api_key: ${NVIDIA_API_KEY}
      temperature: 0.0
      max_tokens: 1024

functions:
    movie_agent:
      _type: react_agent
      tool_names:
        - mcp_movies
      llm_name: nim_llm
      verbose: true
      parse_agent_response_max_retries: 3

workflow:
    _type: langgraph_wrapper
    graph: movie_intent_classifier.py:intent_graph

## Workflow を Telemetry 付きでテストする

これで両方の仕組みが揃って効果を発揮します。先ほどの「Agent の境界を探る」セクションの**同じ 3 つのクエリ** — ドメイン外 2 つとドメイン内 1 つ — を、今度はゲート付き・トレース付きのパイプラインで再生します。別のタブで [http://localhost:6006/](http://localhost:6006/)(`movie-mcp` プロジェクト)を開き、各セルが実行されるたびに新しい trace が現れるのを観察してください。

3 つのベースラインクエリがパイプラインを流れる様子を見たら、自分のクエリも試して agent の境界を探ってみてください — 気になるエッジケース、曖昧な表現、ドメインの内外の境目ぎりぎりの質問など。Phoenix の trace を見れば、classifier がどこに境界を引いたか、そしてなぜそうしたかが正確にわかります。

In [ ]:
!nat run --config_file movie_intent_workflow_tracing.yml --input "How do I cook pasta?"

In [ ]:
!nat run --config_file movie_intent_workflow_tracing.yml --input "What is the capital of France?"

In [ ]:
!nat run --config_file movie_intent_workflow_tracing.yml --input "what is the rating of the movie The Dark Knight Rises?"

### Trace を読む

Phoenix で `movie-mcp` プロジェクトを開き、自分のクエリの trace を 3 つのベースラインクエリと並べて見てください。Trace の形そのものが、classifier が各クエリをどうルーティングしたかを物語ります：

- **拒否されたクエリ**(ドメイン外)は `workflow → classify → reject` の形。LLM 呼び出しが短く 1 回、MCP span はゼロ、レイテンシは1 秒未満。自分のクエリのうちどれがここに来たか確認してみてください — router の判断はあなたの想定と一致していましたか?
- **受理されたクエリ**(ドメイン内)は `workflow → classify → movie_agent → LLM(nim) → tool(search_movies) → MCP` の形。LLM 呼び出しが複数回(分類で 1 回、ReAct ループで 1 回以上)、MCP span が少なくとも 1 回、そしてトークン予算が明らかに大きくなります。

面白いのは曖昧なクエリです — それらの trace を開き、classifier の出力を読んでください。その 1 単語が判断のすべてです。想定外の方向にルーティングされていた場合、修正はほぼ常に agent ではなく classifier の system prompt の方にあります。

## オプション課題:全テーブルをカバーする

構築した `search_movies` tool は **IMDB** テーブルしか触れていません — しかも IMDB の 51 列のうち 6 列しか公開していません。Agent が見えていない情報がまだ大量にあります：

- **`earning`** — 各映画の `Domestic` および `Worldwide` 興行収入
- **`genre`** — 20 ジャンル(Drama、Sci-Fi、Biography、Action 等)にわたる多対多マッピング
- **未公開の IMDB 45 列** — `MetaCritic` スコア、属性別投票の内訳：年齢層別(`VotesU18`、`Votes1829`、`Votes3044`、`Votes45A`)、性別別(`VotesM`、`VotesF`)、地域別(`VotesUS`、`VotesnUS`)、属性別の生票数(`CVotes*`)。

agent は現在、「*Inception の世界興行収入は?*」「*最高評価の伝記映画は?*」「*18 歳未満の視聴者に最も愛されている映画は?*」のような質問には答えられません — どれも tool が公開していないテーブルや列が必要です。このセクションは**オプション**です — 自分でシステムを拡張したい方は試してみてください。

### 狙うサンプル質問

> **「世界興行収入が最も高いドラマは?」**

これに答えるには、agent はすべての `Drama` 映画を見つけ(`genre` テーブル)、世界興行収入を調べ(`earning` テーブル)、勝者をタイトルに解決する(`IMDB` テーブル)必要があります — 現在の tool ではできない 3-way クロス参照です。

### 他に試せる質問

スキーマの異なる部分を引き出すものをいくつか選んでみてください：

| 質問 | 何を引き出すか |
|----------|-------------------|
| *「Inception の世界興行収入は?」* | IMDB (title → id) + `earning` |
| *「最高 IMDB 評価の伝記映画は?」* | IMDB + `genre` |
| *「評価 8.5 以上の最長映画は?」* | IMDB の `Runtime` + `Rating`(注意：`Runtime` は `"148 min"` のようなテキストで格納されている) |
| *「男性視聴者と女性視聴者の評価差が最も大きい映画は?」* | IMDB `VotesM`、`VotesF` |
| *「18 歳未満の視聴者に最も愛されている映画は?」* | IMDB `VotesU18` |
| *「批評家と観客の意見の不一致が最も大きい映画は?」* | IMDB `MetaCritic` 対 `Rating`(MetaCritic は 0-100、Rating は 0-10 であることに注意) |
| *「ROI(世界興行収入 ÷ 制作費)が最高のコメディは?」* | 3 テーブルすべて + 算術 |

各行は、agent が今日持っていない少なくとも 1 つの機能を呼び出すことを強制します — どれを実装するかはあなた次第です。

### 実装ステップ

1. **`MovieDB` を拡張**：`movie_db.py` で `earning` と `genre` を query するメソッドを追加。
2. **新しい tool を登録**：`movie_server.py` で `handle_list_tools` に `types.Tool` エントリを追加(`description` と `inputSchema` の明確さが重要 — LLM がそれを読むので)、`handle_call_tool` にも対応するディスパッチ分岐を追加。
3. **MCP server を再起動**：動作中のプロセスは古い tool リストをキャッシュしているので、`server_process.terminate()` を実行してから `subprocess.Popen` セルを再実行してください。
4. **発見性を検証**：`!nat mcp client tool list --url http://127.0.0.1:8000/mcp/` で新しい tool が説明付きで現れるはずです。
5. **再実行**：上記の質問のいずれかで workflow を再実行し、Phoenix trace を確認します — span ツリーに新しい tool 呼び出しが連鎖して現れ、agent が(ハルシネーションではなく)あなたが配線したクロステーブル推論を実際にやっていることが見えるはずです。

### クリーンアップ

すべてのプロセスを停止します

In [ ]:
# Kill MCP Server
mcp_server_process.terminate()
mcp_server_process.wait()

# Kill Phoenix
phoenix_process.terminate()
phoenix_process.wait()

## ユースケース

NeMo Agent Toolkit は抽象的なフレームワークではありません — 実際の問題を解決する本物のシステムを支える基盤です。それを具体的にするため、NAT の上で動く 2 つの非常に異なるユースケースを見ていきます。同じツールキットが、単一目的の agent から、フル multi-agent のエンタープライズ研究プラットフォームまで、どうスケールするかが見えるはずです。

### AI-Q

**[AI-Q Blueprint](https://github.com/NVIDIA-AI-Blueprints/aiq)** は、組織のデータに接続し、最先端のモデルで推論し、**citation-backed な回答** — もっともらしいだけの文章ではなく — を返すインテリジェントな research agent を構築するための、オープンでエンタープライズ品質のリファレンスです。

NVIDIA NeMo Agent Toolkit と LangGraph の上に構築された最新の [**v2.0**](https://github.com/NVIDIA-AI-Blueprints/aiq/releases/tag/2.0.0) リリース(2026 年 3 月)は、ゼロから書き直された版で、**2 階層の multi-agent アーキテクチャ**を導入しています：入ってきたクエリはまず軽量な **Intent Classifier** に渡り、それがメタ質問か(即座に回答)、**Shallow Research** タスクか(速度重視で境界のある tool 呼び出し agent)、**Deep Research** タスクか(専用の Orchestrator・Planner・Researcher を持つ複数フェーズの workflow、加えて長い実行を始める前にリサーチプランを生成・承認する human-in-the-loop の Clarifier をオプションで付けられる)を判断します。

<div style="text-align: center;">
  <img src="images/aiq_arch.png" style="width: 600px; height: auto;">
</div>


[AI-Q](https://docs.nvidia.com/aiq-blueprint/latest/index.html) を本番品質にしているのは、**NVIDIA NeMo Agent Toolkit (NAT) の直接的でエンドツーエンドな実装**である点です — システムのすべての部分が NAT のプリミティブを通じて定義され、配線され、運用されます。agent、LLM、tool、ルーティングロジックはすべて、環境変数の置換を含む **NAT YAML 設定ファイル**にあるので、(例えば最先端の GPT-5.2 を orchestrator にして、その下にオープンソースの Nemotron researcher を配置するなど)役割の入れ替えを Python のコードに触らずに行えます。

<div style="text-align: center;">
  <img src="images/aiq.png" style="width: 600px; height: auto;">
</div>

ライフサイクル全体が NAT の CLI を通じて実行されます — 対話的な実行に `nat run`、API に `nat serve`、FreshQA と DeepResearch Bench を同梱する組み込み評価ハーネスに `nat eval`。

この NAT 駆動のコアの周りに、AI-Q は自前で構築しなければならなかったエンタープライズの配管を層として乗せています：SSE ストリーミングとイベントリプレイ付きの**非同期 Jobs API**、長時間 deep research ジョブ用の **Dask ベースの分散実行**、**差し替え可能な知識層**(agent コードを変えずに LlamaIndex を NVIDIA の Foundational RAG に切り替え可能)、VLM 経由のマルチモーダル PDF 抽出、ジョブ状態と LangGraph チェックポイント用の **PostgreSQL 永続化**、そして実際に検索結果として得られたソースに対してすべての主張を検証する**決定論的な citation 検証パイプライン**。

結果はこの [blueprint](https://github.com/NVIDIA-AI-Blueprints/aiq) で、現在 [DeepResearch Bench](https://huggingface.co/spaces/muset-ai/DeepResearch-Bench-Leaderboard) と [DeepResearch Bench II](https://agentresearchlab.com/benchmarks/deepresearch-bench-ii/index.html#leaderboard) の両方のリーダーボードで**トップポジション**を維持しつつ、Docker Compose スタック、Helm チャート、Next.js フロントエンド、3 部構成の Jupyter notebook チュートリアルと共に提供されています。DABStep が単一の agent が自分の再利用可能な tool を構築する場合の NAT の姿を見せるとすれば、AI-Q は反対の極 — auth、永続化、observability、評価を備え、実際のユーザーに提供できる multi-agent 研究システム — としての NAT の姿を見せます。

### NVIDIA KGMON Data Explorer

**NVIDIA KGMON Data Explorer** は、NeMo Agent Toolkit の上に構築された実世界のユースケースで、agentic AI の最も難しい問題の 1 つに取り組んでいます：構造化された表形式データに対する多段推論です。ほとんどの「Deep Research」agent は web 検索に依存しますが、金融、ヘルスケア、運用などのドメインでは、答えは CSV、JSON、密度の高いドメインマニュアルの中にあり、インターネット上にはありません。

<div style="text-align: center;">
  <img src="images/data_explorer.png" style="width: 600px; height: auto;">
</div>

チームは [DABStep](https://huggingface.co/spaces/adyen/DABstep) (Data Agent Benchmark for Multi-step Reasoning) でアプローチをベンチマークしました — Adyen の金融決済セクター向け 450 タスクのベンチマークで、84% のタスクが異種データソースとドメインルール間の複雑な相互参照を必要とします。彼らの鍵となる洞察はモデルではなくアーキテクチャに関するものでした：単一の重量級モデルがすべての質問をゼロから解くのではなく、システムを 3 つのフェーズに分割しました — **Learning Loop**(重量級モデルがトレーニングタスクから再利用可能な `helper.py` ライブラリを構築する)、**Fast Inference** ループ(軽量モデルがその事前構築された tool を編成する)、そして **Offline Reflection** フェーズ(重量級モデルが出力をレビューし、洞察を inference プロンプトに注入し直す)。

<div style="text-align: center;">
  <img src="images/data_explorer_arch.png" style="width: 600px; height: auto;">
</div>

結果は：**DABStep で 1 位**、Claude Code + Opus 4.5 ベースラインに対する 30 倍の高速化、そして — 最も印象的なのは — 小さなモデル(Haiku 4.5)が大きなモデル(Opus 4.5)に難しいタスクで **23 ポイントの差で勝利**(89.95 対 66.93)。agent を構築する人にとっての教訓：性能の限界はモデルサイズではなく、アーキテクチャにあることが多いです。詳細は [HuggingFace](https://huggingface.co/blog/nvidia/nemo-agent-toolkit-data-explorer-dabstep-1st-place) のフルブログでお読みください。

## 参考リンク

- [NeMo Agent Toolkit Documentation](https://docs.nvidia.com/nemo/agent-toolkit/latest/index.html)
- [NeMo Agent Toolkit GitHub](https://github.com/NVIDIA/NeMo-Agent-Toolkit)
- [Arize Phoenix](https://phoenix.arize.com/)
- [Arize Phoenix Github](https://github.com/arize-ai/phoenix)

---

## ライセンス

Copyright © 2026 OpenACC-Standard.org. 本資料は OpenACC-Standard.org が NVIDIA Corporation との協力のもと、Creative Commons Attribution 4.0 International (CC BY 4.0) ライセンスのもとで公開しています。本資料には他の団体が開発したハードウェアおよびソフトウェアへの参照が含まれており、該当するすべてのライセンスおよび著作権が適用されます。

<p> <center> <a href="../start_here_ja.ipynb">Home Page</a> </center> </p>

<div>
    <span style="float: left; width: 33%; text-align: left;"><a href="04_langraph_agent_ja.ipynb">Previous Notebook</a></span>
    <span style="float: left; width: 34%; text-align: center;">
        <a href="01_inference_endpoint_ja.ipynb">1</a>
        <a href="02_introduction_mcp_ja.ipynb">2</a>
        <a href="03_low_level_mcp_ja.ipynb">3</a>
        <a href="04_langraph_agent_ja.ipynb">4</a>
        <a >5</a>
        <a href="06_challenge_ja.ipynb">6</a>
        <a href="bonus_challenge/07_bonus_challenge_ja.ipynb">7</a>
    </span>
    <span style="float: left; width: 33%; text-align: right;"><a href="06_challenge_ja.ipynb">Next Notebook</a></span>
</div>